# Pairwise judging with Kaggle models (Task 7)

Create this as a Kaggle **benchmark task** notebook (https://www.kaggle.com/benchmarks/tasks/new), so that
`kaggle_benchmarks` has model-proxy access. Attach a private dataset containing `outputs/eval/*_pairs.jsonl` and
`scripts/judge_pairwise_kaggle.py`. Submitted results came from **File > Save Version > Generate Task** (a server-side
Save & Run All batch run, independent of the browser): version 1 ran the main cell, version 2 the persona re-judge cell.
Download the version output (Output > Download output) and copy `judged/` into `outputs/judged/`.


In [ ]:
import glob, kaggle_benchmarks as kbench
print(glob.glob('/kaggle/input/**/*.py', recursive=True))
print(len(glob.glob('/kaggle/input/**/*_pairs.jsonl', recursive=True)))
print([m for m in kbench.llms.keys() if 'gemini' in m])  # google/gemini-3.6-flash must be listed


In [ ]:
# Live smoke test on 5 pairs (both orders) before the full run
import subprocess, sys, glob
J = glob.glob('/kaggle/input/**/judge_pairwise_kaggle.py', recursive=True)[0]
P = glob.glob('/kaggle/input/**/dpo_uf_vs_sft_alpaca_pairs.jsonl', recursive=True)[0]
r = subprocess.run([sys.executable, J, '--pairs_path', P, '--output_path', '/kaggle/working/test/t_judged.jsonl',
                    '--judge_model', 'google/gemini-3.6-flash', '--limit', '5', '--swap', '--sleep_seconds', '5'],
                   capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])


In [ ]:
# Full run: 10 pair files (all comparisons used in the report), 6 concurrent workers (one file each), each worker sleeps >= 5 s between its requests.
# Resumable: re-running skips (pair, order) items already judged (--reuse_judgments);
# a worker stops early if the daily AI quota runs out, so the next day's run continues from there.
import subprocess, sys, glob, os
from concurrent.futures import ThreadPoolExecutor
J = glob.glob('/kaggle/input/**/judge_pairwise_kaggle.py', recursive=True)[0]
D = os.path.dirname(J)
O = '/kaggle/working/judged'
os.makedirs(O, exist_ok=True)
K = ['dpo_uf_vs_sft_alpaca', 'dpo_hh_lr3e-6_vs_sft_hh', 'dpo_hh_vs_sft_hh',
     'lora_dpo_vs_smollm2-360m-instruct_persona', 'lora_dpo_lr1e-4_ep3_vs_smollm2-360m-instruct_persona',
     'lora_dpo_lr2e-4_ep5_vs_smollm2-360m-instruct_persona', 'dpo_hh_lr3e-6_vs_sft_alpaca', 'dpo_hh_lr1e-5_vs_sft_hh',
     'dpo_uf_vs_sft_hh', 'sft_vs_qwen2.5-0.5b-base_alpaca']
run = lambda k: subprocess.run(
    [sys.executable, J, '--pairs_path', D + '/' + k + '_pairs.jsonl', '--output_path', O + '/' + k + '_judged.jsonl',
     '--judge_model', 'google/gemini-3.6-flash', '--batch_size', '5', '--sleep_seconds', '5', '--max_retries', '3',
     '--reuse_judgments', '--swap', '--reasoning', 'none', '--max_response_chars', '4000'],
    stdout=open(O + '/' + k + '.log', 'w'), stderr=subprocess.STDOUT).returncode
rc = list(ThreadPoolExecutor(6).map(run, K))
print(dict(zip(K, rc)))
os.system('cd /kaggle/working && zip -qr judged.zip judged')
[print(k, open(O + '/' + k + '.log').read().splitlines()[-4:]) for k in K]


In [ ]:
# Persona re-judge (second batch version): only the 3 persona files, reasoning='low', revised style rubric
# (dataset version 3). Results replace the persona files; the first persona judging is archived in
# outputs/judged/persona_reasoning_none/.
import subprocess, sys, glob, os
from concurrent.futures import ThreadPoolExecutor
J = glob.glob('/kaggle/input/**/judge_pairwise_kaggle.py', recursive=True)[0]
D = os.path.dirname(J)
O = '/kaggle/working/judged'
os.makedirs(O, exist_ok=True)
K = ['lora_dpo_vs_smollm2-360m-instruct_persona', 'lora_dpo_lr1e-4_ep3_vs_smollm2-360m-instruct_persona',
     'lora_dpo_lr2e-4_ep5_vs_smollm2-360m-instruct_persona']
run = lambda k: subprocess.run(
    [sys.executable, J, '--pairs_path', D + '/' + k + '_pairs.jsonl', '--output_path', O + '/' + k + '_judged.jsonl',
     '--judge_model', 'google/gemini-3.6-flash', '--batch_size', '5', '--sleep_seconds', '5', '--max_retries', '3',
     '--reuse_judgments', '--swap', '--reasoning', 'low', '--max_response_chars', '4000'],
    stdout=open(O + '/' + k + '.log', 'w'), stderr=subprocess.STDOUT).returncode
print(dict(zip(K, ThreadPoolExecutor(3).map(run, K))))


In [ ]:
# Kaggle 'Generate Task' needs one registered task; it makes no model calls.
import os
def _pa2_task(llm) -> None: kbench.assertions.assert_true(os.path.isdir('/kaggle/working/judged'), expectation='judging produced outputs')
pa2_pairwise_judging = kbench.task(name='pa2_pairwise_judging', description='PA2 pairwise LLM-judge evaluation')(_pa2_task)
pa2_pairwise_judging.run(kbench.llm)
